In [5]:
from pathlib import Path
import pandas as pd

SHEET = "Routine Checkup"

def read_medical_cost_block(xlsx_path: Path) -> pd.DataFrame:
    # Excel row 9 is header -> header=8; keep rows 10–37 -> nrows=28
    df = pd.read_excel(
        xlsx_path,
        sheet_name=SHEET,
        header=8,
        nrows=28,
        engine="openpyxl"
    )

    cols = list(df.columns)
    df = df.rename(columns={
        cols[0]: "group",
        cols[1]: "category",
        cols[2]: "sample_size",
        cols[3]: "yes_pct",
        cols[4]: "yes_ci",
        cols[5]: "no_pct",
        cols[6]: "no_ci",
    })

    df["group"] = df["group"].ffill()
    return df[["group","category","sample_size","yes_pct","yes_ci","no_pct","no_ci"]]

def ci_to_low_high(ci_series: pd.Series) -> pd.DataFrame:
    tokens = ci_series.astype(str).str.extract(
        r"^\(\s*([0-9.]+|\.)\s*-\s*([0-9.]+|\.)\s*\)$"
    )
    low = pd.to_numeric(tokens[0].replace(".", pd.NA), errors="coerce")
    high = pd.to_numeric(tokens[1].replace(".", pd.NA), errors="coerce")
    return pd.DataFrame({"ci_low": low, "ci_high": high})

def to_long_yes_no(df: pd.DataFrame) -> pd.DataFrame:
    yes = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "YES",
        "pct": pd.to_numeric(df["yes_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["yes_ci"]))

    no = pd.DataFrame({
        "group": df["group"],
        "category": df["category"],
        "sample_size": df["sample_size"],
        "response": "NO",
        "pct": pd.to_numeric(df["no_pct"], errors="coerce"),
    }).join(ci_to_low_high(df["no_ci"]))

    return pd.concat([yes, no], ignore_index=True)

def process_one_file(xlsx_path: Path) -> pd.DataFrame:
    base = read_medical_cost_block(xlsx_path)
    long = to_long_yes_no(base)
    long.insert(0, "source_file", xlsx_path.name)
    return long

def process_11_at_once(input_dir: str, output_csv: str, pattern: str="*.xlsx") -> pd.DataFrame:
    in_dir = Path(input_dir)
    files = sorted([p for p in in_dir.glob(pattern) if p.suffix.lower()==".xlsx" and not p.name.startswith("~$")])
    if len(files) == 0:
        raise FileNotFoundError(f"No .xlsx files matched {pattern} in {in_dir}")

    all_long = pd.concat([process_one_file(p) for p in files], ignore_index=True)

    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    all_long.to_csv(out_path, index=False)
    return all_long

# Example:
df_all = process_11_at_once(
    input_dir=r"data/brfss",
    output_csv=r"out/intermediate.csv",
    pattern="*.xlsx"
)
print(df_all.shape)

(616, 8)


In [6]:
import pandas as pd

df = pd.read_csv(
    "out/intermediate.csv",
    header=None,
    names=["file","group","level","n","answer","pct","low","high"]
)

df["file"] = df["file"].str.extract(r"2024_PHR(\d+)_").astype("Int64")

df = df.rename(columns={"file": "PHR"})
df = df.drop(df.index[0])
df.to_csv("out/routine_check.csv", index=False)

In [7]:

print(df.head())

   PHR           group                level    n answer   pct   low  high
1   10           Total                Total  444    YES  73.1  66.9  78.6
2   10             Sex                 Male  224    YES  73.6  66.1  79.9
3   10             Sex               Female  220    YES  72.6  61.9  81.2
4   10  Race/Ethnicity  White, Non-Hispanic   89    YES  75.5  52.8  89.5
5   10  Race/Ethnicity  Black, Non-Hispanic    N    YES   NaN   NaN   NaN
